# The Complete Transformer Block

In previous notebooks, we built the individual pieces. Now we'll assemble them into a complete **transformer block** — the repeating unit that makes transformers work.

In this notebook, we'll:

1. Implement **layer normalization** from scratch
2. Understand **residual connections** (skip connections)
3. Build the **feed-forward network** (FFN)
4. Assemble a complete **transformer block**
5. **Stack multiple blocks** to create a mini transformer

**Prerequisites:** Complete notebooks [01](./01_attention_mechanisms.ipynb), [02](./02_multi_head_attention.ipynb), and [03](./03_positional_encoding.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")

## Part 1: Reusable Components from Previous Notebooks

In [ ]:
# === Reusable functions from previous notebooks ===

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    weights = softmax(scores)
    return weights @ V, weights


def sinusoidal_positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            PE[pos, i] = np.sin(pos / denom)
            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos / denom)
    return PE


print("Building blocks loaded!")

## Part 2: Layer Normalization

### Why Normalize?

As data passes through many layers, numbers can grow very large or very small. This makes training unstable — like trying to balance a pencil on its tip.

**Layer normalization** rescales each vector to have:
- Mean ≈ 0
- Standard deviation ≈ 1

Think of it like **grading on a curve**: raw scores [95, 30, 72, 88] become standardized grades.

In [ ]:
class LayerNorm:
    """Layer Normalization — normalizes each vector independently."""
    
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)    # Learnable scale (initialized to 1)
        self.beta = np.zeros(d_model)    # Learnable shift (initialized to 0)
        self.eps = eps                    # Small number to prevent division by zero
    
    def __call__(self, x):
        # Compute mean and std for each word vector
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        
        # Normalize
        x_norm = (x - mean) / (std + self.eps)
        
        # Scale and shift (learnable parameters)
        return self.gamma * x_norm + self.beta


# Demonstrate layer normalization
d_model = 8
ln = LayerNorm(d_model)

# Before normalization: numbers are all over the place
x = np.array([
    [150.0, -200.0, 3.0, 50.0, 80.0, -10.0, 25.0, 100.0],
    [0.001, 0.002, 0.003, 0.001, 0.002, 0.001, 0.003, 0.002],
])

x_normed = ln(x)

print("Before LayerNorm:")
print(f"  Word 1: [{', '.join(f'{v:>8.1f}' for v in x[0])}]")
print(f"    mean = {x[0].mean():.1f}, std = {x[0].std():.1f}")
print(f"  Word 2: [{', '.join(f'{v:>8.4f}' for v in x[1])}]")
print(f"    mean = {x[1].mean():.4f}, std = {x[1].std():.4f}")
print()
print("After LayerNorm:")
print(f"  Word 1: [{', '.join(f'{v:>8.3f}' for v in x_normed[0])}]")
print(f"    mean = {x_normed[0].mean():.6f}, std = {x_normed[0].std():.3f}")
print(f"  Word 2: [{', '.join(f'{v:>8.3f}' for v in x_normed[1])}]")
print(f"    mean = {x_normed[1].mean():.6f}, std = {x_normed[1].std():.3f}")
print()
print("Both vectors now have mean ≈ 0 and std ≈ 1, regardless of original scale!")

In [ ]:
# Visualize the effect of layer normalization
np.random.seed(42)
x_demo = np.random.randn(10, 32)

# Simulate what happens after many layers (numbers grow)
for _ in range(5):
    W = np.random.randn(32, 32) * 0.5
    x_demo = x_demo @ W

x_demo_normed = LayerNorm(32)(x_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([x_demo[i] for i in range(10)])
axes[0].set_title('Before LayerNorm\n(values exploded after many layers)', fontsize=13)
axes[0].set_xlabel('Word position', fontsize=11)
axes[0].set_ylabel('Value', fontsize=11)

axes[1].boxplot([x_demo_normed[i] for i in range(10)])
axes[1].set_title('After LayerNorm\n(stable, centered values)', fontsize=13)
axes[1].set_xlabel('Word position', fontsize=11)
axes[1].set_ylabel('Value', fontsize=11)

plt.tight_layout()
plt.show()

print("Without LayerNorm, values grow out of control.")
print("With LayerNorm, values stay in a stable range — training is much easier!")

## Part 3: Residual Connections (Skip Connections)

A **residual connection** adds the input of a layer directly to its output:

```
output = layer(x) + x    ← the "+ x" is the residual connection
```

### The Cooking Analogy

Imagine you're making a sauce. At each step, you add a new ingredient. Without residual connections, if one step goes wrong (too much salt), you've ruined everything. With residual connections, you keep a backup of the sauce before each step — even if one step is bad, the original flavors survive.

In [ ]:
# Demonstrate residual connections

def simulate_layers(x, n_layers, use_residual=False):
    """Simulate passing through multiple layers."""
    d = x.shape[-1]
    history = [np.linalg.norm(x)]
    
    for i in range(n_layers):
        # Simulate a layer (random transformation)
        W = np.random.randn(d, d) * 0.5
        layer_output = np.tanh(x @ W)
        
        if use_residual:
            x = layer_output + x  # Residual: add input back!
        else:
            x = layer_output       # No residual: input is lost
        
        history.append(np.linalg.norm(x))
    
    return x, history


# Compare with and without residual connections
np.random.seed(42)
x_input = np.random.randn(5, 32)

np.random.seed(42)
_, history_no_res = simulate_layers(x_input.copy(), 20, use_residual=False)

np.random.seed(42)
_, history_with_res = simulate_layers(x_input.copy(), 20, use_residual=True)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(history_no_res, 'r-o', label='Without Residual', markersize=4, linewidth=2)
ax.plot(history_with_res, 'b-o', label='With Residual', markersize=4, linewidth=2)
ax.set_xlabel('Layer Number', fontsize=13)
ax.set_ylabel('Signal Magnitude (norm)', fontsize=13)
ax.set_title('Effect of Residual Connections Through 20 Layers', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Without residuals: signal can shrink (vanish) or explode")
print("With residuals: signal stays stable because original input is always preserved")
print("\nThis is why deep transformers (96+ layers) can work!")

## Part 4: Feed-Forward Network (FFN)

After attention lets words talk to each other, the FFN gives each word **private thinking time**.

It's a simple two-layer neural network applied to each word independently:
1. **Expand** the vector (e.g., 512 → 2048)
2. Apply **activation function** (ReLU or GELU)
3. **Compress** back (2048 → 512)

In [ ]:
def relu(x):
    """ReLU activation: max(0, x)"""
    return np.maximum(0, x)


def gelu(x):
    """GELU activation (used in modern transformers like GPT)."""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


class FeedForward:
    """Feed-Forward Network: expand → activate → compress."""
    
    def __init__(self, d_model, d_ff, activation='relu'):
        self.W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
        self.activation = relu if activation == 'relu' else gelu
        self.d_model = d_model
        self.d_ff = d_ff
    
    def __call__(self, x):
        # Step 1: Expand (d_model → d_ff)
        hidden = x @ self.W1 + self.b1
        
        # Step 2: Activation
        hidden = self.activation(hidden)
        
        # Step 3: Compress (d_ff → d_model)
        output = hidden @ self.W2 + self.b2
        
        return output


# Demonstrate
d_model = 8
d_ff = 32  # 4× expansion (typical: d_ff = 4 × d_model)

ffn = FeedForward(d_model, d_ff, activation='gelu')

x = np.random.randn(3, d_model)  # 3 words × 8 dims
out = ffn(x)

print(f"Feed-Forward Network:")
print(f"  Input:  {x.shape}  ({d_model} dimensions)")
print(f"  Hidden: (3, {d_ff})  ({d_ff} dimensions — 4× expansion!)")
print(f"  Output: {out.shape}  ({d_model} dimensions — back to original!)")
print(f"\n  The expansion to {d_ff} dims gives the model more 'space to think'.")
print(f"  Each word is processed INDEPENDENTLY (no interaction between words here).")

In [ ]:
# Visualize ReLU vs GELU
x_range = np.linspace(-3, 3, 200)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_range, relu(x_range), 'r-', linewidth=2.5, label='ReLU (original transformer)')
ax.plot(x_range, gelu(x_range), 'b-', linewidth=2.5, label='GELU (GPT, BERT, modern)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Input', fontsize=13)
ax.set_ylabel('Output', fontsize=13)
ax.set_title('Activation Functions Used in Transformers', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ReLU: sharp cutoff at 0 (simple and fast)")
print("GELU: smooth curve (slightly better performance, used in modern models)")
print("Both make the network non-linear — without them, stacking layers would be useless.")

## Part 5: The Complete Transformer Block

Now let's assemble everything into a single transformer block!

```
Input
  │
  ├──────────────────┐
  ▼                  │
LayerNorm            │  (residual connection)
  │                  │
Multi-Head Attention │
  │                  │
  + ◄────────────────┘  (add input back)
  │
  ├──────────────────┐
  ▼                  │
LayerNorm            │  (residual connection)
  │                  │
Feed-Forward         │
  │                  │
  + ◄────────────────┘  (add input back)
  │
Output
```

In [ ]:
class MultiHeadAttention:
    """Multi-Head Attention (from notebook 02)."""
    
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_K = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_V = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_O = np.random.randn(d_model, d_model) * scale
    
    def __call__(self, x, mask=None):
        head_outputs = []
        all_weights = []
        
        for h in range(self.n_heads):
            Q = x @ self.W_Q[h]
            K = x @ self.W_K[h]
            V = x @ self.W_V[h]
            out, w = scaled_dot_product_attention(Q, K, V, mask)
            head_outputs.append(out)
            all_weights.append(w)
        
        concatenated = np.concatenate(head_outputs, axis=-1)
        return concatenated @ self.W_O, all_weights


class TransformerBlock:
    """
    One complete transformer block (Pre-LayerNorm variant).
    
    Architecture:
        x → LayerNorm → Multi-Head Attention → + residual
          → LayerNorm → Feed-Forward          → + residual
    """
    
    def __init__(self, d_model, n_heads, d_ff):
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, activation='gelu')
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_ff = d_ff
    
    def __call__(self, x, mask=None):
        # Sub-layer 1: Multi-Head Attention with residual connection
        x_norm = self.ln1(x)                     # Layer norm
        attn_out, attn_weights = self.attention(x_norm, mask)  # Attention
        x = x + attn_out                          # Residual connection
        
        # Sub-layer 2: Feed-Forward with residual connection
        x_norm = self.ln2(x)                     # Layer norm
        ffn_out = self.ffn(x_norm)               # Feed-forward
        x = x + ffn_out                           # Residual connection
        
        return x, attn_weights


# Test a single transformer block
d_model = 16
n_heads = 4
d_ff = 64  # 4× expansion

block = TransformerBlock(d_model, n_heads, d_ff)

# Input: 5 words, 16 dimensions each
sentence = ["The", "cat", "sat", "on", "mat"]
x = np.random.randn(len(sentence), d_model)

output, attn_weights = block(x)

print(f"Transformer Block Configuration:")
print(f"  d_model (embedding size): {d_model}")
print(f"  n_heads:                  {n_heads}")
print(f"  d_k (per head):           {d_model // n_heads}")
print(f"  d_ff (FFN inner dim):     {d_ff}")
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {output.shape}  ← same as input!")
print(f"\nThis means blocks can be STACKED — the output of one becomes")
print(f"the input of the next. Transformers typically stack 6-96 blocks.")

## Part 6: Stacking Multiple Blocks

Real transformers stack many blocks. Each block refines the representation:
- **Early blocks**: simple patterns (grammar, nearby words)
- **Middle blocks**: combining information
- **Later blocks**: complex patterns (meaning, reasoning)

In [ ]:
class MiniTransformer:
    """A complete (tiny) transformer: embeddings + positional encoding + N blocks."""
    
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len=100):
        self.d_model = d_model
        self.n_layers = n_layers
        
        # Embedding table (word → vector)
        self.embedding = np.random.randn(vocab_size, d_model) * 0.02
        
        # Positional encoding
        self.pos_encoding = sinusoidal_positional_encoding(max_len, d_model)
        
        # Stack of transformer blocks
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        
        # Final layer norm
        self.final_ln = LayerNorm(d_model)
    
    def forward(self, token_ids, mask=None):
        seq_len = len(token_ids)
        
        # Step 1: Token embeddings
        x = self.embedding[token_ids]  # Look up each token
        
        # Step 2: Add positional encoding
        x = x + self.pos_encoding[:seq_len]
        
        # Step 3: Pass through each transformer block
        all_attention = []
        for block in self.blocks:
            x, attn = block(x, mask)
            all_attention.append(attn)
        
        # Step 4: Final layer norm
        x = self.final_ln(x)
        
        return x, all_attention


# Create a mini transformer
vocab_size = 100
d_model = 32
n_heads = 4
d_ff = 128
n_layers = 4

transformer = MiniTransformer(vocab_size, d_model, n_heads, d_ff, n_layers)

# Simulate processing a sentence
sentence = ["The", "cat", "sat", "on", "the", "mat"]
token_ids = np.array([10, 42, 67, 23, 10, 55])  # Fake token IDs

output, all_attention = transformer.forward(token_ids)

print(f"Mini Transformer Configuration:")
print(f"  Vocabulary size:  {vocab_size}")
print(f"  d_model:          {d_model}")
print(f"  n_heads:          {n_heads}")
print(f"  d_ff:             {d_ff}")
print(f"  n_layers:         {n_layers}")
print(f"")
print(f"Input:  {len(token_ids)} tokens")
print(f"Output: {output.shape}  ({len(token_ids)} words × {d_model} dims)")
print(f"")
print(f"Total parameter count (approximate):")

# Count parameters
n_params = vocab_size * d_model  # Embeddings
per_block = (3 * n_heads * d_model * (d_model // n_heads)  # Q, K, V projections
             + d_model * d_model   # W_O
             + d_model * d_ff + d_ff  # FFN layer 1
             + d_ff * d_model + d_model  # FFN layer 2
             + 2 * 2 * d_model)   # 2 LayerNorms
n_params += per_block * n_layers
print(f"  ~{n_params:,} parameters")
print(f"  (Real GPT-3 has 175,000,000,000 — 175 billion!)")

## Part 7: Visualize Attention Across Layers

Let's see how attention patterns evolve through the layers:

In [ ]:
# Visualize attention from all layers (showing head 0 from each layer)
fig, axes = plt.subplots(1, n_layers, figsize=(18, 4.5))

for layer_idx in range(n_layers):
    ax = axes[layer_idx]
    # Show head 0 from each layer
    w = all_attention[layer_idx][0]  # Layer's head 0
    
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.8)
    ax.set_xticks(range(len(sentence)))
    ax.set_yticks(range(len(sentence)))
    ax.set_xticklabels(sentence, fontsize=9, rotation=45, ha='right')
    ax.set_yticklabels(sentence, fontsize=9)
    ax.set_title(f'Layer {layer_idx + 1}\n(Head 1)', fontsize=12)
    
    if layer_idx == 0:
        ax.set_ylabel('Query (FROM)', fontsize=11)

fig.suptitle('Attention Patterns Across Layers (Head 1 of each layer)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("In a trained model, you would see:")
print("  Early layers: local patterns (nearby words, punctuation)")
print("  Middle layers: syntactic patterns (subject-verb, modifier-noun)")
print("  Later layers: semantic patterns (meaning, coreference)")
print("\nOur model is randomly initialized, so patterns are not meaningful yet.")
print("After training, the patterns become interpretable!")

## Part 8: The Complete Architecture Diagram

Let's trace the full data flow and verify shapes at each step:

In [ ]:
print("""Complete Transformer Data Flow
================================

Input: "The cat sat on the mat" (6 tokens)
""")

# Trace through each step
token_ids = np.array([10, 42, 67, 23, 10, 55])
seq_len = len(token_ids)

# Step 1: Embedding
x = transformer.embedding[token_ids]
print(f"Step 1: Token Embedding")
print(f"  token_ids → embedding table lookup")
print(f"  Shape: {token_ids.shape} → {x.shape}  (6 tokens × 32 dims)")

# Step 2: Positional encoding
pe = transformer.pos_encoding[:seq_len]
x = x + pe
print(f"\nStep 2: + Positional Encoding")
print(f"  x = word_embedding + position_encoding")
print(f"  Shape: {x.shape}  (unchanged — position is ADDED)")

# Step 3: Transformer blocks
for i, block in enumerate(transformer.blocks):
    x_before = x.copy()
    
    # Sub-layer 1: Attention
    x_norm = block.ln1(x)
    attn_out, _ = block.attention(x_norm)
    x = x + attn_out
    
    # Sub-layer 2: FFN
    x_norm = block.ln2(x)
    ffn_out = block.ffn(x_norm)
    x = x + ffn_out
    
    print(f"\nStep 3.{i+1}: Transformer Block {i+1}")
    print(f"  LayerNorm → Multi-Head Attn → + residual → LayerNorm → FFN → + residual")
    print(f"  Shape: {x.shape}  (always {seq_len} × {d_model})")

# Step 4: Final layer norm
x = transformer.final_ln(x)
print(f"\nStep 4: Final Layer Norm")
print(f"  Shape: {x.shape}")

print(f"\n{'=' * 50}")
print(f"Final output: {x.shape}")
print(f"Each word now has a rich, context-aware representation.")
print(f"\nTo make predictions (next word, classification, etc.),")
print(f"you'd add a task-specific head on top:")
print(f"  - Language model: Linear(d_model → vocab_size) + softmax")
print(f"  - Classifier: Linear(d_model → num_classes) + softmax")

## Part 9: Parameter Count Breakdown

Let's understand where all the parameters live in a transformer:

In [ ]:
configs = [
    ("Our Mini Model", 100, 32, 4, 128, 4),
    ("GPT-2 Small", 50257, 768, 12, 3072, 12),
    ("BERT Base", 30522, 768, 12, 3072, 12),
    ("GPT-3", 50257, 12288, 96, 49152, 96),
]

print(f"{'Model':<16} {'Vocab':>8} {'d_model':>8} {'Heads':>6} {'d_ff':>6} "
      f"{'Layers':>7} {'Total Params':>15}")
print("─" * 75)

for name, vocab, dm, heads, dff, layers in configs:
    # Embeddings
    emb_params = vocab * dm
    
    # Per block
    attn_params = 4 * dm * dm  # W_Q, W_K, W_V, W_O (simplified)
    ffn_params = 2 * dm * dff  # Two linear layers
    ln_params = 4 * dm         # 2 layer norms × 2 params each
    block_params = attn_params + ffn_params + ln_params
    
    total = emb_params + block_params * layers
    
    if total >= 1e9:
        total_str = f"{total/1e9:.1f}B"
    elif total >= 1e6:
        total_str = f"{total/1e6:.1f}M"
    else:
        total_str = f"{total/1e3:.1f}K"
    
    print(f"{name:<16} {vocab:>8,} {dm:>8} {heads:>6} {dff:>6} "
          f"{layers:>7} {total_str:>15}")

print("\nWhere do the parameters live?")
print("  ~30% in attention (W_Q, W_K, W_V, W_O)")
print("  ~65% in feed-forward networks (two large linear layers)")
print("  ~5%  in embeddings and layer norms")
print("\nThe FFN is actually the biggest component by parameter count!")

## Summary

We built a complete transformer from scratch! Here's everything we assembled:

```
Token IDs → Embedding + Positional Encoding → [Transformer Block × N] → Output

Each Transformer Block:
  ┌─ LayerNorm → Multi-Head Attention → + residual ─┐
  └─ LayerNorm → Feed-Forward Network → + residual ─┘
```

### Components Recap

| Component | Purpose | Analogy |
|-----------|---------|--------|
| Embedding | Words → numbers | Dictionary lookup |
| Positional Encoding | Add position info | Seat numbers |
| Multi-Head Attention | Words talk to each other | Group discussion |
| Feed-Forward Network | Each word thinks privately | Quiet reflection |
| Layer Normalization | Keep numbers stable | Grading on a curve |
| Residual Connection | Preserve original signal | Keeping a backup |

### Key Takeaways

1. A **transformer block** = attention + FFN + layer norm + residuals
2. Blocks can be **stacked** because output shape = input shape
3. **Residual connections** prevent information loss in deep networks
4. **Layer normalization** keeps training stable
5. **FFN** is where most parameters live (~65%)
6. The whole architecture is **parallelizable** — that's why GPUs love transformers

### What's Next?

With this foundation, you're ready to:
- Implement a transformer in PyTorch
- Understand how GPT, BERT, and other models are structured
- Move on to [fine-tuning](../../02-fine-tuning/) to adapt pre-trained transformers